# IMDB 二分类：早停与模型保存

对应 Keras 版本：`Keras应用/第四章-神经网络入门：分类与回归/IMDB二分类_早停与模型保存.ipynb`

本实验在 PyTorch 中实现：
1. 自定义 EarlyStopping 回调
2. 自定义 ModelCheckpoint 保存最佳模型
3. 加载保存的模型进行评估

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"PyTorch: {torch.__version__}")

## 1. 回调类定义

对应 Keras 的 `keras.callbacks.EarlyStopping` 和 `keras.callbacks.ModelCheckpoint`。

In [ ]:
class Callback:
    """回调函数基类"""
    def on_epoch_end(self, epoch, logs=None): pass


class EarlyStopping(Callback):
    """
    早停回调。
    对应 Keras: keras.callbacks.EarlyStopping
    """
    def __init__(self, monitor="val_loss", patience=2, mode="min"):
        self.monitor = monitor
        self.patience = patience
        self.mode = mode
        self.counter = 0
        self.best_value = None
        self.should_stop = False
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None:
            self.best_value = current
        elif (self.mode == "min" and current < self.best_value) or \
             (self.mode == "max" and current > self.best_value):
            self.best_value = current
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  [EarlyStopping] {self.monitor} 在 {self.patience} 轮内未改善，停止训练")


class ModelCheckpoint(Callback):
    """
    模型检查点。
    对应 Keras: keras.callbacks.ModelCheckpoint
    """
    def __init__(self, filepath, monitor="val_loss", mode="min"):
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best_value = None
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None or \
           (self.mode == "min" and current < self.best_value) or \
           (self.mode == "max" and current > self.best_value):
            self.best_value = current
            torch.save({
                'epoch': epoch,
                'model_state_dict': logs['model'].state_dict(),
                self.monitor: current,
            }, self.filepath)
            print(f"  [Checkpoint] 模型已保存 (val_loss={current:.4f})")


print("回调类定义完成")

## 2. 数据准备

In [ ]:
# 加载 IMDB 数据
from tensorflow.keras.datasets import imdb
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = np.asarray(train_labels, dtype=np.float32)
y_test = np.asarray(test_labels, dtype=np.float32)

# 划分验证集
x_val = torch.from_numpy(x_train[:10000])
y_val = torch.from_numpy(y_train[:10000])
x_train_t = torch.from_numpy(x_train[10000:])
y_train_t = torch.from_numpy(y_train[10000:])

print(f"训练集: {x_train_t.shape[0]}, 验证集: {x_val.shape[0]}, 测试集: {x_test.shape[0]}")

## 3. 模型定义

In [ ]:
class IMDBClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10000, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x)).squeeze(-1)

model = IMDBClassifier()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

## 4. 训练与回调

In [ ]:
def train_epoch(model, x, y, optimizer, loss_fn, batch_size=512):
    model.train()
    total_loss = 0
    correct = 0
    n = len(x)
    indices = torch.randperm(n)
    x, y = x[indices], y[indices]
    
    for i in range(0, n, batch_size):
        bx, by = x[i:i+batch_size], y[i:i+batch_size]
        optimizer.zero_grad()
        out = model(bx)
        loss = loss_fn(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
        correct += ((out >= 0.5) == by).sum().item()
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, x, y, loss_fn):
    model.eval()
    out = model(x)
    loss = loss_fn(out, y)
    correct = ((out >= 0.5) == y).sum().item()
    return loss.item(), correct / len(y)


# 使用回调训练
checkpoint_path = "imdb_best.pt"
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, mode="min"),
    ModelCheckpoint(filepath=checkpoint_path, monitor="val_loss", mode="min")
]

epochs = 20
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, x_train_t, y_train_t, optimizer, loss_fn)
    val_loss, val_acc = evaluate(model, x_val, y_val, loss_fn)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    logs = {
        "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc, "model": model
    }
    
    print(f"Epoch {epoch+1:2d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    
    for cb in callbacks:
        cb.on_epoch_end(epoch, logs)
    
    if any(cb.should_stop for cb in callbacks if hasattr(cb, 'should_stop')):
        break

## 5. 加载最佳模型

In [ ]:
# 加载保存的最佳模型
checkpoint = torch.load(checkpoint_path, weights_only=False)

# 重建模型并加载权重
best_model = IMDBClassifier()
best_model.load_state_dict(checkpoint['model_state_dict'])

print(f"[成功] 最佳模型已加载 (val_loss={checkpoint['val_loss']:.4f})")

## 6. 测试集评估

In [ ]:
x_test_t = torch.from_numpy(x_test)
y_test_t = torch.from_numpy(y_test)

test_loss, test_acc = evaluate(best_model, x_test_t, y_test_t, loss_fn)
print(f"最佳模型的测试准确率: {test_acc:.4f}")

## 7. 清理

In [ ]:
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print(f"已清理临时文件: {checkpoint_path}")